In [ ]:
#This project models the maintenance and editing of municipal voting precinct boundaries.
#Each edit to a precinct represents a meaningful business event involving:

    #A precinct (the geographic unit being updated)

    #A user/editor (the person making the change)

    #A timestamp (when the change occurred)

    #A spatial footprint (area and perimeter of the precinct)

#The dimensional data mart enables historical analysis of this process, including:

    #Which precincts were edited

    #Who performed the edits

    #When edits occurred

    #How spatial characteristics vary across precincts

In [ ]:
#Several transformations were applied to prepare the data for dimensional modeling:

    #Cleaned and standardized column names

    #Converted timestamps to MySQL‑compatible formats

    #Derived date attributes (year, month, day, week, quarter)

    #Split the original CSV into multiple logical dimensions

    #Integrated MongoDB metadata with CSV precinct data
    
    #Created surrogate keys for all dimensions

    #Built a date dimension from last_edited_date

    #Constructed a fact table representing precinct edit events

In [ ]:
#  1.0. Import Libraries
#This section imports all Python libraries required for ETL, SQL connectivity, and MongoDB access.
# Got data from https://opendata.charlottesville.org/datasets/21fad2a83e4c44f58e1c3fcb99756c23_12/explore?location=38.040050%2C-78.485050%2C13&showTable=true
# Split it into CSV's and JSON's.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import pymongo
import json
import certifi

In [2]:
#  2.0. Define Connection Variables
#Here I define the MySQL and MongoDB connection parameters used throughout the notebook.

In [3]:
mysql_args = {
    "uid" : "root",
    "pwd" : "1234",
    "hostname" : "localhost",
    "dbname" : "midterm1"
}

mongodb_args = {
    "user_name" : "linj716",
    "password" : "lQkiz4ibmrJg56aC",
    "cluster_name" : "sandbox",
    "cluster_subnet" : "hkabmqp",
    "cluster_location" : "atlas",
    "db_name" : "midterm1_nosql"
}

In [4]:
#  3.0. Drop & Recreate the MySQL Data Warehouse
#Before loading new data, I drop the existing midterm1 database (if it exists) and recreate it. 

In [5]:
server_engine = create_engine(
    f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}"
)
conn = server_engine.connect()

conn.execute(text("DROP DATABASE IF EXISTS midterm1;"))
conn.execute(text("CREATE DATABASE midterm1;"))

conn.close()


In [6]:
#  4.0. Define Helper Functions
#These functions replicate the patterns from Labs 03a and 04:

In [7]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

In [8]:
# 📌 5.0. Extract Phase
#In this section, I extract data from:
    #The file Voting_Precinct_Area.csv was retrieved from the local file system.

    #This file was split into two logical datasets:

        #precincts.csv — precinct names and identifiers

        #areas.csv — spatial attributes (area, perimeter)

    #Both were extracted using Pandas.


In [9]:
# Extract precincts (business keys + names)
precincts_df = pd.read_csv("precincts.csv")
precincts_df

,OBJECTID,PrecinctName,PrecinctNumber
0,1,Jackson-Via,301
1,2,Buford,303
2,3,Summit/Clark,102
3,4,Key Recreation,101
4,5,CHS,401
5,6,Walker,402
6,7,Johnson,302
7,8,Carver,201
8,9,Trailblazer/Venable,202


In [10]:
# Extract areas (geometry attributes)
areas_df = pd.read_csv("areas.csv")
areas_df

,OBJECTID,ShapeSTArea,ShapeSTLength
0,1,3.204707e+07,36433.312214
1,2,1.944308e+07,20543.635260
2,3,2.560258e+07,25249.436475
3,4,3.739865e+07,30639.345844
4,5,6.531807e+07,56452.490090
5,6,4.667657e+07,36297.361836
6,7,2.038958e+07,25029.699453
7,8,2.409965e+07,22265.210029
8,9,1.569274e+07,26276.312910


In [ ]:
#The file metadata.json was loaded into a MongoDB collection.
#Using PyMongo, the project extracted:

    #created_user

    #created_date

    #last_edited_user

    #last_edited_date

#This data became the dim_metadata dimension.

In [11]:
client = get_mongo_client(**mongodb_args)

data_dir = os.path.join(os.getcwd(), "data")
json_files = {"metadata": "metadata.json"}

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)

client = get_mongo_client(**mongodb_args)
metadata_df = get_mongo_dataframe(client, mongodb_args["db_name"], "metadata", {})
metadata_df

,OBJECTID,created_user,created_date,last_edited_user,last_edited_date
0,1,None,None,WINKLERM,2023/09/26 17:48:37+00
1,2,None,None,WINKLERM,2023/09/26 17:37:05+00
2,3,None,None,CITY,2024/06/18 18:13:32+00
3,4,None,None,WINKLERM,2024/01/27 08:12:56+00
4,5,None,None,WINKLERM,2023/09/26 17:37:05+00
5,6,None,None,WINKLERM,2023/09/26 17:37:05+00
6,7,None,None,WINKLERM,2023/09/26 17:37:05+00
7,8,None,None,WINKLERM,2023/09/26 17:37:05+00
8,9,None,None,CITY,2024/06/18 18:13:56+00


In [12]:
# 📌 6.0. Load DataFrames into MySQL
#Using the set_dataframe() function, I load:

#dim_products

#dim_stores

#dim_customers

#fact_sales

In [ ]:
#The transformed datasets were loaded into a MySQL data mart:

#dim_precincts, dim_areas, dim_metadata, and dim_date were loaded using set_dataframe()

#fact_precincts was created directly in MySQL using a SQL CREATE TABLE AS SELECT statement

In [13]:
set_dataframe(precincts_df, "dim_precincts", "precinct_key", "insert", **mysql_args)

set_dataframe(areas_df, "dim_areas", "area_key", "insert", **mysql_args)

set_dataframe(metadata_df, "dim_metadata", "metadata_key", "insert", **mysql_args)


In [16]:
engine = create_engine(
    f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}/{mysql_args['dbname']}"
)
conn = engine.connect()

conn.execute(text("""
    CREATE TABLE fact_precincts AS
    SELECT 
        m.OBJECTID,
        d.date_key
    FROM dim_metadata m
    JOIN dim_date d
        ON DATE(m.last_edited_date) = d.date;
"""))

conn.execute(text("""
    ALTER TABLE fact_precincts
    ADD fact_key INT AUTO_INCREMENT PRIMARY KEY FIRST;
"""))

conn.close()


In [14]:
# 📌 7.0. Transform Phase
#Convert transaction_date to MySQL DATE then built the Date Dimension

In [15]:
engine = create_engine(
    f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}/{mysql_args['dbname']}"
)
conn = engine.connect()

conn.execute(text("""
    UPDATE dim_metadata
    SET last_edited_date = STR_TO_DATE(last_edited_date, '%Y/%m/%d %H:%i:%s+00');
"""))

conn.execute(text("""
    CREATE TABLE dim_date AS
    SELECT DISTINCT
        DATE(last_edited_date) AS date,
        YEAR(last_edited_date) AS year,
        MONTH(last_edited_date) AS month,
        DAY(last_edited_date) AS day,
        WEEK(last_edited_date) AS week,
        QUARTER(last_edited_date) AS quarter
    FROM dim_metadata
    WHERE last_edited_date IS NOT NULL;
"""))

conn.execute(text("""
    ALTER TABLE dim_date
    ADD date_key INT AUTO_INCREMENT PRIMARY KEY FIRST;
"""))

conn.close()

df_dim_date = get_sql_dataframe("SELECT * FROM dim_date LIMIT 5;", **mysql_args)
df_dim_date


,date_key,date,year,month,day,week,quarter
0,1,2023-09-26,2023,9,26,39,3
1,2,2024-06-18,2024,6,18,24,2
2,3,2024-01-27,2024,1,27,3,1


In [ ]:
#  8.0. Star Schema Join
#This demonstrates that the warehouse schema is correct and all foreign keys match.

In [17]:
df_joined = get_sql_dataframe("""
SELECT 
    f.fact_key,
    dp.precinct_key,
    da.area_key,
    dm.metadata_key,
    dd.date_key,
    dp.PrecinctName,
    dp.PrecinctNumber,
    da.ShapeSTArea,
    da.ShapeSTLength,
    dm.last_edited_user,
    dm.last_edited_date,
    dd.year,
    dd.month,
    dd.week
FROM fact_precincts f
JOIN dim_precincts dp ON f.OBJECTID = dp.OBJECTID
JOIN dim_areas da ON f.OBJECTID = da.OBJECTID
JOIN dim_metadata dm ON f.OBJECTID = dm.OBJECTID
JOIN dim_date dd ON f.date_key = dd.date_key;
""", **mysql_args)

df_joined.head()


,fact_key,precinct_key,area_key,metadata_key,date_key,PrecinctName,PrecinctNumber,ShapeSTArea,ShapeSTLength,last_edited_user,last_edited_date,year,month,week
0,1,1,1,1,1,Jackson-Via,301,3.204707e+07,36433.312214,WINKLERM,2023-09-26 17:48:37,2023,9,39
1,2,2,2,2,1,Buford,303,1.944308e+07,20543.635260,WINKLERM,2023-09-26 17:37:05,2023,9,39
2,3,3,3,3,2,Summit/Clark,102,2.560258e+07,25249.436475,CITY,2024-06-18 18:13:32,2024,6,24
3,4,4,4,4,3,Key Recreation,101,3.739865e+07,30639.345844,WINKLERM,2024-01-27 08:12:56,2024,1,3
4,5,5,5,5,1,CHS,401,6.531807e+07,56452.490090,WINKLERM,2023-09-26 17:37:05,2023,9,39


In [ ]:
# 9.0. Analysis Queries

In [18]:
df_area_by_precinct = get_sql_dataframe("""
SELECT 
    dp.PrecinctName,
    dp.PrecinctNumber,
    da.ShapeSTArea AS area
FROM fact_precincts f
JOIN dim_precincts dp ON f.OBJECTID = dp.OBJECTID
JOIN dim_areas da ON f.OBJECTID = da.OBJECTID
ORDER BY area DESC;
""", **mysql_args)

df_area_by_precinct.head()

,PrecinctName,PrecinctNumber,area
0,CHS,401,6.531807e+07
1,Walker,402,4.667657e+07
2,Key Recreation,101,3.739865e+07
3,Jackson-Via,301,3.204707e+07
4,Summit/Clark,102,2.560258e+07


In [19]:
df_edits_by_user = get_sql_dataframe("""
SELECT 
    dm.last_edited_user,
    COUNT(*) AS edit_count
FROM fact_precincts f
JOIN dim_metadata dm ON f.OBJECTID = dm.OBJECTID
GROUP BY dm.last_edited_user
ORDER BY edit_count DESC;
""", **mysql_args)

df_edits_by_user.head()

,last_edited_user,edit_count
0,WINKLERM,7
1,CITY,2


In [20]:
df_edits_by_date = get_sql_dataframe("""
SELECT 
    dd.date,
    COUNT(*) AS precincts_edited
FROM fact_precincts f
JOIN dim_date dd ON f.date_key = dd.date_key
GROUP BY dd.date
ORDER BY dd.date;
""", **mysql_args)

df_edits_by_date.head()


,date,precincts_edited
0,2023-09-26,6
1,2024-01-27,1
2,2024-06-18,2


In [21]:
df_area_by_date = get_sql_dataframe("""
SELECT 
    dd.date,
    SUM(da.ShapeSTArea) AS total_area
FROM fact_precincts f
JOIN dim_areas da ON f.OBJECTID = da.OBJECTID
JOIN dim_date dd ON f.date_key = dd.date_key
GROUP BY dd.date
ORDER BY dd.date;
""", **mysql_args)

df_area_by_date.head()


,date,total_area
0,2023-09-26,2.079740e+08
1,2024-01-27,3.739865e+07
2,2024-06-18,4.129532e+07


In [ ]:
# 10.0 Extracting from SQL Table
#The project extracted from MySQL tables using get_sql_dataframe().
#This demonstrates relational extraction as required.

#Tables extracted include:

    #dim_precincts

    #dim_areas

    #dim_metadata

    #dim_date

    #fact_precincts

In [23]:
df_dim_precincts_extract = get_sql_dataframe("SELECT * FROM dim_precincts;", **mysql_args)
df_dim_areas_extract = get_sql_dataframe("SELECT * FROM dim_areas;", **mysql_args)
df_dim_metadata_extract = get_sql_dataframe("SELECT * FROM dim_metadata;", **mysql_args)
df_dim_date_extract = get_sql_dataframe("SELECT * FROM dim_date;", **mysql_args)
df_fact_precincts_extract = get_sql_dataframe("SELECT * FROM fact_precincts;", **mysql_args)


In [24]:
df_dim_precincts_extract.head()

,precinct_key,OBJECTID,PrecinctName,PrecinctNumber
0,1,1,Jackson-Via,301
1,2,2,Buford,303
2,3,3,Summit/Clark,102
3,4,4,Key Recreation,101
4,5,5,CHS,401


In [25]:
df_dim_areas_extract.head()

,area_key,OBJECTID,ShapeSTArea,ShapeSTLength
0,1,1,3.204707e+07,36433.312214
1,2,2,1.944308e+07,20543.635260
2,3,3,2.560258e+07,25249.436475
3,4,4,3.739865e+07,30639.345844
4,5,5,6.531807e+07,56452.490090


In [26]:
df_dim_metadata_extract.head()

,metadata_key,OBJECTID,created_user,created_date,last_edited_user,last_edited_date
0,1,1,None,None,WINKLERM,2023-09-26 17:48:37
1,2,2,None,None,WINKLERM,2023-09-26 17:37:05
2,3,3,None,None,CITY,2024-06-18 18:13:32
3,4,4,None,None,WINKLERM,2024-01-27 08:12:56
4,5,5,None,None,WINKLERM,2023-09-26 17:37:05


In [27]:
df_dim_date_extract.head()

,date_key,date,year,month,day,week,quarter
0,1,2023-09-26,2023,9,26,39,3
1,2,2024-06-18,2024,6,18,24,2
2,3,2024-01-27,2024,1,27,3,1


In [28]:
df_fact_precincts_extract.head()

,fact_key,OBJECTID,date_key
0,1,1,1
1,2,2,1
2,3,3,2
3,4,4,3
4,5,5,1


In [ ]:
#   Overall, this project successfully demonstrates the complete design and implementation of a multi‑source ETL pipeline and dimensional data mart. 
#   I extracted data from three different systems—a CSV file from the local file system, 
#   a JSON metadata file loaded into MongoDB, and relational tables stored in MySQL—then transformed, 
#   cleaned, and integrated these sources into a unified analytical model. 
#   The data was reshaped into four dimensions (precincts, areas, metadata, and date) and one fact table representing precinct edit events, 
#   forming a clear and efficient star schema. After loading the data mart into MySQL, 
#   I validated the schema through multi‑table joins and executed analytical SQL queries that combined the fact table with multiple dimensions, 
#   using aggregation and grouping to answer meaningful questions about editing activity, spatial characteristics, and temporal patterns. 
#   The results confirm that the pipeline functions end‑to‑end, the schema supports historical analysis, 
#   and the system meets all requirements for a functional, well‑designed data mart.